In [56]:
# Place synthetic samples into the dataset

import os
import shutil
import subprocess
import pandas as pd
from PIL import Image

In [57]:
singan_datasets_folder = '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning_num_images_5_self_supervised'

In [58]:
RANDOM_SAMPLES_PY = '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/random_samples.py'

In [59]:
SELF_SUPERVISED = True
RANDOM_SAMPLES_FOLDER = '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output/RandomSamples' if not SELF_SUPERVISED else '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output/FinetuningSamples/RandomSamples'

In [60]:
import os
import random
import shutil
from collections import deque

In [61]:
# Validate files in finetuning folder

In [62]:
mask_image_counts = []
for dataset in os.listdir(singan_datasets_folder):
    expansion_factor = float(dataset.split('expansion_factor_')[-1])
    dataset_path = os.path.join(singan_datasets_folder, dataset)
    for cv_subject in os.listdir(dataset_path):
        finetuning_folder_path = os.path.join(dataset_path, cv_subject, 'finetuning')
        files = os.listdir(finetuning_folder_path)
        num_image_files = 0
        num_mask_files = 0
        for f in files:
            if 'mask' in f:
                num_mask_files += 1
            else:
                num_image_files += 1
        mask_image_counts.append({
            'dataset': dataset,
            'cv_subject': cv_subject,
            'num_image_files': num_image_files,
            'num_mask_files': num_mask_files
        })
        print(f"Dataset: {dataset}, CV Subject: {cv_subject}, Images: {num_image_files}, Masks: {num_mask_files}")
        assert num_image_files == num_mask_files, f"Mismatch in number of image and mask files in {finetuning_folder_path}"

Dataset: augmented_dataset_expansion_factor_2.0, CV Subject: FD-030, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_2.0, CV Subject: FD-027, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_2.0, CV Subject: FD-029, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_2.0, CV Subject: FD-032, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_2.0, CV Subject: FD-031, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_8.0, CV Subject: FD-030, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_8.0, CV Subject: FD-027, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_8.0, CV Subject: FD-029, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_8.0, CV Subject: FD-032, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_8.0, CV Subject: FD-031, Images: 5, Masks: 5
Dataset: augmented_dataset_expansion_factor_1.0, CV Subject: FD-030, Images: 5, Masks: 5
Dataset: augmented_da

In [63]:
for dataset in os.listdir(singan_datasets_folder):
    expansion_factor = float(dataset.split('expansion_factor_')[-1])
    dataset_path = os.path.join(singan_datasets_folder, dataset)
    for cv_subject in os.listdir(dataset_path):
        finetuning_augmented_folder_path = os.path.join(dataset_path, cv_subject, 'finetuning_augmented')
        # Clear finetuning augmented folder if it exists
        if os.path.exists(finetuning_augmented_folder_path):
            shutil.rmtree(finetuning_augmented_folder_path)
        os.makedirs(finetuning_augmented_folder_path, exist_ok=True)

In [64]:
CONSTANT_EXPANSION_FACTOR = 10 # Set to None if you want to use the expansion factor from the dataset name

In [65]:
file_transfers = []
images_missing_samples = []
for dataset in os.listdir(singan_datasets_folder):
    expansion_factor = CONSTANT_EXPANSION_FACTOR if CONSTANT_EXPANSION_FACTOR else float(dataset.split('expansion_factor_')[-1])
    dataset_path = os.path.join(singan_datasets_folder, dataset)
    for cv_subject in os.listdir(dataset_path):
        finetuning_folder_path = os.path.join(dataset_path, cv_subject, 'finetuning')
        finetuning_augmented_folder_path = os.path.join(dataset_path, cv_subject, 'finetuning_augmented')
        # Clear finetuning augmented folder if it exists
        if os.path.exists(finetuning_augmented_folder_path):
            shutil.rmtree(finetuning_augmented_folder_path)
        os.makedirs(finetuning_augmented_folder_path, exist_ok=True)

        # 1) Gather and shuffle generated images per original file
        finetuning_image_paths_per_file = {}
        for fn in os.listdir(finetuning_folder_path):
            if 'image' not in fn:
                continue
            base_name = os.path.splitext(fn)[0]
            matching = [d for d in os.listdir(RANDOM_SAMPLES_FOLDER) if base_name in d]
            if not matching:
                print(f"No matching random samples found for {fn} in {RANDOM_SAMPLES_FOLDER}. Skipping.")
                images_missing_samples.append({
                    'file_name': fn,
                    'basename': base_name,
                    'cv_subject': cv_subject,
                    'dataset': dataset
                })
                continue
            image_dir = os.path.join(RANDOM_SAMPLES_FOLDER, matching[0], 'gen_start_scale=0')
            paths = [
                os.path.join(image_dir, f)
                for f in os.listdir(image_dir)
                if '_img' in f
            ]
            if paths:
                random.shuffle(paths)
                finetuning_image_paths_per_file[fn] = paths

        # 2) Compute how many new samples we need total
        dataset_expansion_amount = (len(os.listdir(finetuning_folder_path)) / 2) * expansion_factor

        # 3) Cycle through files, picking one image at a time from each
        selected_images = []
        queue = deque(finetuning_image_paths_per_file.keys())

        while len(selected_images) < dataset_expansion_amount and queue:
            fn = queue.popleft()
            paths = finetuning_image_paths_per_file[fn]
            if paths:
                # take the next image
                selected_images.append(paths.pop(0))
                # if there are more for this file, re‑enqueue it
                if paths:
                    queue.append(fn)

        # 4) Copy exactly the selected images
        for image_path in selected_images:
            mask_path = image_path.replace('_img.png', '_mask.png')

            original_image_name = image_path.split('/')[-3]
            image_basename = original_image_name + '-' + os.path.basename(image_path).replace('_img.png', '_image.png')

            mask_basename = image_basename.replace('_image.png', '_mask.png')

            print(image_basename, mask_basename)

            # Copy the image path
            shutil.copy(image_path, os.path.join(finetuning_augmented_folder_path, image_basename))
            file_transfers.append({
                'original_path': image_path,
                'new_path': os.path.join(finetuning_augmented_folder_path, image_basename   )
            })
            # Copy the mask path
            shutil.copy(mask_path, os.path.join(finetuning_augmented_folder_path, mask_basename))
            file_transfers.append({
                'original_path': mask_path,
                'new_path': os.path.join(finetuning_augmented_folder_path, mask_basename)
            })

        # 5) Copy all images in finetuning into finetuning_augmented
        for fn in os.listdir(finetuning_folder_path):
            if fn.endswith('image.png') or fn.endswith('mask.png'):
                shutil.copy(os.path.join(finetuning_folder_path, fn), os.path.join(finetuning_augmented_folder_path, fn))
                file_transfers.append({
                    'original_path': os.path.join(finetuning_folder_path, fn),
                    'new_path': os.path.join(finetuning_augmented_folder_path, fn)
                })

FD-030-slice-27-image-7_image.png FD-030-slice-27-image-7_mask.png
FD-030-slice-28-image-38_image.png FD-030-slice-28-image-38_mask.png
FD-030-slice-29-image-47_image.png FD-030-slice-29-image-47_mask.png
FD-030-slice-26-image-0_image.png FD-030-slice-26-image-0_mask.png
FD-030-slice-30-image-6_image.png FD-030-slice-30-image-6_mask.png
FD-030-slice-27-image-1_image.png FD-030-slice-27-image-1_mask.png
FD-030-slice-28-image-27_image.png FD-030-slice-28-image-27_mask.png
FD-030-slice-29-image-27_image.png FD-030-slice-29-image-27_mask.png
FD-030-slice-26-image-40_image.png FD-030-slice-26-image-40_mask.png
FD-030-slice-30-image-33_image.png FD-030-slice-30-image-33_mask.png
FD-030-slice-27-image-11_image.png FD-030-slice-27-image-11_mask.png
FD-030-slice-28-image-33_image.png FD-030-slice-28-image-33_mask.png
FD-030-slice-29-image-46_image.png FD-030-slice-29-image-46_mask.png
FD-030-slice-26-image-1_image.png FD-030-slice-26-image-1_mask.png
FD-030-slice-30-image-47_image.png FD-030-sl

In [89]:
pd.DataFrame(images_missing_samples)['cv_subject'].value_counts()

cv_subject
FD-027    70
FD-030    21
FD-029    21
FD-031    21
FD-032    14
Name: count, dtype: int64

In [87]:
pd.DataFrame(images_missing_samples)

,file_name,basename,cv_subject,dataset
0,FD-030-slice-31-mask.png,FD-030-slice-31-mask,FD-030,augmented_dataset_expansion_factor_2.0
1,FD-030-slice-35-image.png,FD-030-slice-35-image,FD-030,augmented_dataset_expansion_factor_2.0
2,FD-030-slice-32-mask.png,FD-030-slice-32-mask,FD-030,augmented_dataset_expansion_factor_2.0
3,FD-030-slice-28-mask.png,FD-030-slice-28-mask,FD-030,augmented_dataset_expansion_factor_2.0
4,FD-030-slice-34-mask.png,FD-030-slice-34-mask,FD-030,augmented_dataset_expansion_factor_2.0
...,...,...,...,...
492,FD-031-slice-23-image.png,FD-031-slice-23-image,FD-031,augmented_dataset_expansion_factor_16.0
493,FD-031-slice-20-mask.png,FD-031-slice-20-mask,FD-031,augmented_dataset_expansion_factor_16.0
494,FD-031-slice-24-mask.png,FD-031-slice-24-mask,FD-031,augmented_dataset_expansion_factor_16.0
495,FD-031-slice-19-image.png,FD-031-slice-19-image,FD-031,augmented_dataset_expansion_factor_16.0


In [74]:
file_transfers[1]

{'original_path': '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-32-image/gen_start_scale=0/17_mask.png',
 'new_path': '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning_num_images_10/augmented_dataset_expansion_factor_2.0/FD-030/finetuning_augmented/FD-030-slice-32-image-17_mask.png'}

In [69]:
pd.DataFrame(file_transfers)['original_path'].values.tolist()

['/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-32-image/gen_start_scale=0/1_img.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-32-image/gen_start_scale=0/1_mask.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-27-image/gen_start_scale=0/56_img.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-27-image/gen_start_scale=0/56_mask.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-31-image/gen_start_scale=0/41_img.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-31-image/gen_start_scale=0/41_mask.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-28-image/gen_start_scale

In [70]:
pd.DataFrame(file_transfers)['new_path'].values.tolist()

['/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning_num_images_10/augmented_dataset_expansion_factor_2.0/FD-030/finetuning_augmented/1_image.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning_num_images_10/augmented_dataset_expansion_factor_2.0/FD-030/finetuning_augmented/FD-030-slice-32-image-1_mask.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning_num_images_10/augmented_dataset_expansion_factor_2.0/FD-030/finetuning_augmented/56_image.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning_num_images_10/augmented_dataset_expansion_factor_2.0/FD-030/finetuning_augmented/FD-030-slice-27-image-56_mask.png',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning_num_images_10/augmented_dataset_expa

In [ ]:
mask_path = image_path.replace('_img.png', '_mask.png')

In [60]:
original_image_name = image_path.split('/')[-3]

In [ ]:
image_basename = original_image_name + '-' + os.path.basename(image_path)
mask_basename = image_basename.replace('_img.png', '_mask.png')

In [64]:
image_basename

'FD-031-slice-21-image-15_img.png'

In [65]:
image_path

'/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-031-slice-21-image/gen_start_scale=0/15_img.png'

In [8]:
os.path.join(finetuning_augmented_folder_path, fn)

'/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_with_finetuning/augmented_dataset_expansion_factor_16.0/FD-031/finetuning_augmented/FD-031-slice-19-image.png'

In [36]:
os.listdir('/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/')

['FD-029-slice-06-image',
 'FD-032-slice-07-image',
 'FD-029-slice-66-image',
 'FD-030-slice-18-image',
 'FD-029-slice-24-image',
 'FD-030-slice-68-image',
 'FD-032-slice-65-image',
 'FD-029-slice-20-image',
 'FD-029-slice-71-image',
 'FD-029-slice-04-image',
 'FD-027-slice-04-image',
 'FD-030-slice-27-image',
 'FD-031-slice-10-image',
 'FD-027-slice-60-image',
 'FD-029-slice-17-image',
 'FD-032-slice-19-image',
 'FD-032-slice-10-image',
 'FD-029-slice-12-image',
 'FD-027-slice-02-image',
 'FD-029-slice-19-image',
 'FD-031-slice-65-image',
 'FD-032-slice-01-image',
 'FD-031-slice-67-image',
 'FD-030-slice-03-image',
 'FD-032-slice-18-image',
 'FD-032-slice-22-image',
 'FD-029-slice-18-image',
 'FD-030-slice-26-image',
 'FD-030-slice-25-image',
 'FD-029-slice-07-image',
 'FD-030-slice-19-image',
 'FD-032-slice-68-image',
 'FD-029-slice-11-image',
 'FD-029-slice-31-image',
 'FD-032-slice-61-image',
 'FD-030-slice-15-image',
 'FD-032-slice-67-image',
 'FD-032-slice-20-image',
 'FD-027-sli

In [30]:
finetuning_image_paths

['/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-27-image/gen_scale_start=0',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-28-image/gen_scale_start=0',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-29-image/gen_scale_start=0',
 '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output_benchmark/RandomSamples/FD-030-slice-26-image/gen_scale_start=0']

In [21]:
fn

'FD-030-slice-27-image.png'

In [23]:
os.listdir(RANDOM_SAMPLES_FOLDER)

['FD-029-slice-06-image',
 'FD-032-slice-07-image',
 'FD-029-slice-66-image',
 'FD-030-slice-18-image',
 'FD-029-slice-24-image',
 'FD-030-slice-68-image',
 'FD-032-slice-65-image',
 'FD-029-slice-20-image',
 'FD-029-slice-71-image',
 'FD-029-slice-04-image',
 'FD-027-slice-04-image',
 'FD-030-slice-27-image',
 'FD-031-slice-10-image',
 'FD-027-slice-60-image',
 'FD-029-slice-17-image',
 'FD-032-slice-19-image',
 'FD-032-slice-10-image',
 'FD-029-slice-12-image',
 'FD-027-slice-02-image',
 'FD-029-slice-19-image',
 'FD-031-slice-65-image',
 'FD-032-slice-01-image',
 'FD-031-slice-67-image',
 'FD-030-slice-03-image',
 'FD-032-slice-18-image',
 'FD-032-slice-22-image',
 'FD-029-slice-18-image',
 'FD-030-slice-26-image',
 'FD-030-slice-25-image',
 'FD-029-slice-07-image',
 'FD-030-slice-19-image',
 'FD-032-slice-68-image',
 'FD-029-slice-11-image',
 'FD-029-slice-31-image',
 'FD-032-slice-61-image',
 'FD-030-slice-15-image',
 'FD-032-slice-67-image',
 'FD-032-slice-20-image',
 'FD-027-sli